# **RAG (Retrival Augmented Generation)** **SYSTEMS**

In [1]:
!pip install chromadb sentence-transformers groq pandas -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [3]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os

print("All Libraries Imported Successfully")

All Libraries Imported Successfully


In [5]:
GROQ_API_KEY = "*******************************************"

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client Initialized")

Groq client Initialized


In [13]:
df = pd.read_csv("college_notes.csv")

print("Shape of dataset: ", df.shape)
print("\nColumns names: ", df.columns.tolist())
print("\nFirst 5 rows: ")
print(df.head())

Shape of dataset:  (14, 4)

Columns names:  ['note_id', 'subject', 'topic', 'content']

First 5 rows: 
  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N006  Machine Learning       Supervised Learning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Supervised learning is a type of machine learn...  


In [14]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample pf topics:")
print(df[['note_id', 'subject', 'topic']].to_string(index=False))

print("\nLength of content (number of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset:
subject
Machine Learning      5
Data Engineering      4
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample pf topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python Programming                 Pandas Library
   N015 Python

In [15]:
documents = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID: {ids[0]}")
print(f"First document metadata: {metadatas[0]}")
print(f"First document content: {documents[0][:100]}...")

Total chunks prepared: 14
First document ID: note_N001
First document metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document content: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [17]:
print("Loading embedding model......")
print("This may take 30-60 seconds on first run -model is being downloaded")
print("(Subsequent runs will be faster)")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print(f"Test embedding shape: {embedding_model.encode(['This is a test sentence']).shape}")

Loading embedding model......
This may take 30-60 seconds on first run -model is being downloaded
(Subsequent runs will be faster)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully.
Test embedding shape: (1, 384)


In [18]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="college_notes_rag")

print("ChromaDB client created.")
print(f"Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

ChromaDB client created.
Collection name: college_notes_rag
Documents in collection so far: 0


In [19]:
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")

collection.add(
    documents=documents,
    embeddings=embeddings,
    metadatas=metadatas,
    ids=ids
)

print(f"\nDocuments in collection now: {collection.count()}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (14, 384)

Documents in collection now: 14


In [20]:
def retrieve_relevant_chunks(question, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )

  return results

print("Retrieval function defined Successfully.")
print("Function: retrieve_relevant_chunks(question, top_k=3)")

Retrieval function defined Successfully.
Function: retrieve_relevant_chunks(question, top_k=3)


In [23]:
test_question = "What is ETL and how does it work in data engineering?"

results = retrieve_relevant_chunks(test_question)

for i, (doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
    print(f"\nResult {i+1}:")
    print(f" Subject : {meta['subject']}")
    print(f" Topic : {meta['topic']}")
    print(f" Distance : {dist:.4f}")
    print(f" Content : {doc[:100]}...")


Result 1:
 Subject : Data Engineering
 Topic : ETL Pipelines
 Distance : 0.2269
 Content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...

Result 2:
 Subject : Data Engineering
 Topic : APIs and Data Collection
 Distance : 1.0690
 Content : An API or Application Programming Interface allows two software applications to talk to each other. ...

Result 3:
 Subject : Python Programming
 Topic : Data Visualization
 Distance : 1.3375
 Content : Data visualization is the process of representing data as charts graphs and visual formats. Python l...


In [ ]:
def build_from_results(results):
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
      chunk_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}]\n{doc}"

In [6]:
def generate_rag_answer(question, context):
  system_prompt = """ You are a helpful assistant for engineering students.
  You will be given context retrieved from a college knowledge base, ans a student's question

  RULES:
  1. Answer ONLY using the information provided in the context below.
  2. If the answer is not found in the context, say exactly:
    "I don't have enough information in my knowledge base to answer your question."
  3. Do not use your general training knowledge.
  4. Keep answers clear, accurate, and beginner-friendly.
  5. Mention which source of information came from when possible."""

  user_prompt = f"""Context from Knowledge Base:
  {context}

  ---

  Student Question:
  {question}
  Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant",

      messages = [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature=0.1,
      max_tokens=500
  )

  answer = response.choices[0].message.content

  return answer

print("RAG function defined Successfully.")
print("Function: generate_rag_answer(question, context)")

RAG function defined Successfully.
Function: generate_rag_answer(question, context)


In [5]:
def ask_college_assistant(question, top_k=3, verbose=True):
  if verbose:
    print(f"Question: {question}")

  results = retrieve_relevant_chunks(question, top_k=top_k)